# Read Avro with rugo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/avro.ipynb)

Rugo **reads** Avro object container files - the format Kafka sinks, Hadoop-era data lakes and Iceberg manifests are written in - with no PyArrow, pandas or NumPy. **It doesn't write Avro**: it's a reader only. The last section times it against fastavro and the Apache `avro` package.

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.avro"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.avro", "space_missions.avro")
print(f"space_missions.avro: {os.path.getsize('space_missions.avro'):,} bytes")

## Read the schema, not the data

An Avro file carries its writer schema in its header. `read_metadata` returns it, with the codec, without decoding a record.

In [ ]:
from rugo import avro

meta = avro.read_metadata("space_missions.avro")
print("codec:", meta.codec)
for field in meta.schema["fields"]:
    print(f"{field['name']:16} {field['type']}")

`["null", "double"]` is how Avro spells a nullable column. `Rocket` is a nested record holding a name and an enum.

## Read the whole file

`read_avro` is a context manager that yields **morsels** - batches of typed columns, made of whole Avro blocks. Logical types are decoded (the launch time arrives as a timestamp), enums arrive as their symbol, and a nested record arrives as JSON text.

In [ ]:
with avro.read_avro("space_missions.avro") as reader:
    for morsel in reader:
        print(morsel)

## Read only the fields you need

`columns` picks fields. A dotted name reads one field out of a nested record as an ordinary column. Avro has no statistics, so there's no `predicates` argument - filter the morsel's values instead.

In [ ]:
with avro.read_avro(
    "space_missions.avro",
    columns=["Company", "Mission", "Rocket.Name", "Rocket.Status"],
) as reader:
    morsel = next(iter(reader))

print(morsel)
active = [name for name, status in zip(morsel.column("Rocket.Name").to_pylist(),
                                       morsel.column("Rocket.Status").to_pylist())
          if status == "Active"]
print(len(set(active)), "rockets still active")

## Read through a reader schema

A reader schema is the shape you want, matched against the shape the file was written with: fields it leaves out are dropped, fields the file doesn't have take their default, and numbers can widen (int to long, float to double). It's how Avro reads old and new versions of a schema together.

In [ ]:
reader_schema = {
    "type": "record",
    "name": "Launch",
    "fields": [
        {"name": "Mission", "type": "string"},
        {"name": "Price", "type": ["null", "double"], "default": None},
        {"name": "Source", "type": "string", "default": "space_missions"},
    ],
}

with avro.read_avro("space_missions.avro", reader_schema=reader_schema) as reader:
    print(next(iter(reader)))

## Compare with fastavro and the Apache `avro` package

Rugo doesn't write Avro, so fastavro writes a 100,000-row test file: 20 fields, deflate-compressed, in the mix of an event stream. Each reader then reads it twice - every field, and just 3 fields (rugo is given `columns`; the others are given a reader schema of those fields, which is how they skip the rest).

rugo returns typed columns, while fastavro and `avro` return a Python dict per record, so rugo is also timed with every column converted to Python values (`rugo+pylist`) - the fair comparison if you want Python objects at the end.

In [ ]:
%pip install -q fastavro avro

In [ ]:
import datetime, decimal, io, random
import fastavro

WORDS = ["alpha", "bravo", "charlie", "delta", "echo", "foxtrot", "golf", "hotel"]
schema = {"type": "record", "name": "Event", "fields": [
    {"name": "id", "type": "long"}, {"name": "user_id", "type": "long"},
    {"name": "small", "type": "int"}, {"name": "amount", "type": "double"},
    {"name": "ratio", "type": "float"}, {"name": "flag", "type": "boolean"},
    {"name": "event", "type": "string"},
    {"name": "country", "type": {"type": "enum", "name": "Country", "symbols": ["AU", "DE", "GB", "JP", "US"]}},
    {"name": "name", "type": "string"}, {"name": "url", "type": "string"},
    {"name": "note", "type": ["null", "string"]}, {"name": "score", "type": ["double", "null"]},
    {"name": "created_at", "type": {"type": "long", "logicalType": "timestamp-micros"}},
    {"name": "event_date", "type": {"type": "int", "logicalType": "date"}},
    {"name": "price", "type": {"type": "bytes", "logicalType": "decimal", "precision": 12, "scale": 2}},
    {"name": "session", "type": {"type": "fixed", "name": "Session", "size": 16}},
    {"name": "counter", "type": ["null", "long"]},
    {"name": "device", "type": {"type": "record", "name": "Device", "fields": [
        {"name": "os", "type": "string"}, {"name": "model", "type": ["null", "string"]}]}},
    {"name": "tags", "type": {"type": "array", "items": "string"}},
    {"name": "metrics", "type": {"type": "map", "values": "double"}},
]}

def records(n, rnd=random.Random(42)):
    start = datetime.datetime(2026, 1, 1, tzinfo=datetime.timezone.utc)
    for i in range(n):
        yield {"id": i, "user_id": rnd.randrange(5_000_000), "small": rnd.randrange(-1000, 1000),
               "amount": rnd.uniform(0, 10_000), "ratio": rnd.random(), "flag": rnd.random() < 0.3,
               "event": rnd.choice(["click", "view", "purchase"]),
               "country": rnd.choice(["AU", "DE", "GB", "JP", "US"]),
               "name": " ".join(rnd.choice(WORDS) for _ in range(rnd.randint(1, 4))),
               "url": f"https://example.com/{rnd.choice(WORDS)}/{rnd.randrange(1_000_000)}",
               "note": None if rnd.random() < 0.7 else rnd.choice(WORDS),
               "score": None if rnd.random() < 0.1 else rnd.gauss(50, 15),
               "created_at": start + datetime.timedelta(seconds=i),
               "event_date": start.date() + datetime.timedelta(days=rnd.randrange(365)),
               "price": decimal.Decimal(rnd.randrange(-10_000_000, 100_000_000)).scaleb(-2),
               "session": rnd.randbytes(16), "counter": None if rnd.random() < 0.5 else rnd.randrange(1 << 40),
               "device": {"os": rnd.choice(["ios", "android"]), "model": None if rnd.random() < 0.2 else "m1"},
               "tags": [rnd.choice(WORDS) for _ in range(rnd.randint(0, 3))],
               "metrics": {rnd.choice(WORDS): rnd.random() for _ in range(rnd.randint(0, 3))}}

buffer = io.BytesIO()
fastavro.writer(buffer, fastavro.parse_schema(schema), records(100_000), codec="deflate", sync_interval=64_000)
data = buffer.getvalue()
print(f"{len(data):,} bytes")

In [ ]:
import json, time
from avro.datafile import DataFileReader
from avro.io import DatumReader
from avro.schema import parse

NARROW = ["id", "amount", "country"]
narrow_schema = {**schema, "fields": [f for f in schema["fields"] if f["name"] in NARROW]}

def rugo_read(columns, to_python=False):
    with avro.read_avro(data, columns=columns) as reader:
        for morsel in reader:
            if to_python:
                for name in morsel.column_names:
                    morsel.column(name).to_pylist()

def fastavro_read(columns):
    for _ in fastavro.reader(io.BytesIO(data), reader_schema=narrow_schema if columns else None):
        pass

def apache_read(columns):
    readers_schema = parse(json.dumps(narrow_schema)) if columns else None
    for _ in DataFileReader(io.BytesIO(data), DatumReader(readers_schema=readers_schema)):
        pass

readers = {
    "rugo": lambda c: rugo_read(c),
    "rugo+pylist": lambda c: rugo_read(c, to_python=True),
    "fastavro": fastavro_read,
    "avro (Apache)": apache_read,
}

def best_of(fn, columns, runs):
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        fn(columns)
        times.append(time.perf_counter() - start)
    return min(times)

for label, columns in (("every field", None), ("3 fields", NARROW)):
    print(label)
    base = best_of(readers["rugo"], columns, 3)
    for name, fn in readers.items():
        seconds = base if name == "rugo" else best_of(fn, columns, 1 if name.startswith("avro") else 3)
        print(f"  {name:14} {seconds * 1000:9.1f} ms   {seconds / base:6.1f}x rugo")

Timings depend on the machine - on Colab you share one with other notebooks, so expect some noise. The published comparison, measured on a quiet machine with a script you can run yourself, is at [rugo.dev](https://rugo.dev/#performance).